## Test script for cross datasets training

In [1]:
import numpy as np

from scaleflow.data import AnnDataLocation, DataManager, GroupedDistribution, prepare_datasets, split_datasets
from scaleflow.data._dataloader import CombinedSampler, InMemorySampler, ReservoirSampler, ValidationSampler
from scaleflow.datasets import sample_adata
from scaleflow.model import ScaleFlow
from scaleflow.training import Metrics
import scanpy as sc


/home/icb/xiaotong.fu/miniconda3/envs/pancellflow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata1 = sc.read_h5ad('/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/combosciplex_with_embeddings.h5ad')
adata2 = sc.read_h5ad('/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe_with_embeddings.h5ad')
adata1.obs["control"] = (
    (adata1.obs["drug_0"] == "control") &
    (adata1.obs["drug_1"] == "control")
)
adata2.obs["control"] = (
    (adata2.obs["drug_0"] == "control") &
    (adata2.obs["drug_1"] == "control")
)

In [3]:
adl = AnnDataLocation()
data_manager = DataManager(
    dist_flag_key="control",
    src_dist_keys=["cell_line"],
    tgt_dist_keys=["drug_0", "drug_1"],
    rep_keys={
        "cell_line": "cell_line_embeddings",
        "drug_0": "drug_0_embeddings",
        "drug_1": "drug_1_embeddings",
    },
    data_location=adl.obsm["X_state"],
)

In [4]:

# Prepare data
print("Preparing data...")
gd1 = data_manager.prepare_data(adata1)
gd2 = data_manager.prepare_data(adata2)

# Split datasets
print("Splitting datasets...")
data = split_datasets({"gd1": gd1, "gd2": gd2}, split_by=["drug_0","drug_1"], split_key="split", ratios=[0.4, 0.3,0.3], random_state=42, holdout_combinations=False)
train_splits = {k:v["train"] for k,v in data.items()}
val_splits = {k:v["val"] for k,v in data.items()}
ds1, ds2 = train_splits["gd1"], train_splits["gd2"]
# Create training samplers
print("Creating samplers...")
sampler = CombinedSampler(
    samplers={
        "gd1": InMemorySampler(gd1, np.random.default_rng(43), batch_size=8),
        "gd2": InMemorySampler(gd2, np.random.default_rng(44), batch_size=8),
    },
    rng=np.random.default_rng(42),
)

# Create validation sampler - returns all conditions at once (finite, not infinite)

val_samplers = {
    "gd1": ValidationSampler(
        val_splits["gd1"],  # Use one split for validation
        n_conditions_on_log_iteration=3,  # Limit to 5 conditions for faster testing
        n_conditions_on_train_end=3,
        seed=42,
    ),
    "gd2": ValidationSampler(
        val_splits["gd2"],  # Use one split for validation
        n_conditions_on_log_iteration=3,  # Limit to 5 conditions for faster testing
        n_conditions_on_train_end=3,
        seed=42,
    ),
}



# Initialize sampler
print("Initializing sampler...")
sampler.init_sampler()

# Sample a batch
print("Sampling batch...")
sample_batch = sampler.sample()
print(f"Condition shapes: {[(k, v.shape) for k, v in sample_batch['condition'].items()]}")

# Create and prepare model
print("Creating model...")
sf = ScaleFlow()

print("Preparing model...")
sf.prepare_model(
    sample_batch=sample_batch,
    max_combination_length=1,  # Single perturbations
)

Preparing data...
Splitting datasets...
Creating samplers...
Initializing sampler...
Sampling batch...
Condition shapes: [('cell_line', (1, 1, 1)), ('drug_0', (1, 1, 200)), ('drug_1', (1, 1, 200))]
Creating model...
Preparing model...


In [ ]:

# Train
print("Training...")
sf.train(
    val_dataloader=val_samplers,
    train_dataloader=sampler,
    num_iterations=10000,
    valid_freq=500,
    callbacks=[Metrics(["e_distance"])],  
    monitor_metrics=["e_distance"] 
)

print("Done!")

Training...


  0%|          | 0/10000 [00:00<?, ?it/s]

  5%|▌         | 501/10000 [01:07<19:58,  7.93it/s] 


Starting validation on 2 dataset(s)...
